# AbstractTensors — Tensors, Metrics & Permutation Symmetries

This notebook walks through:
1. Defining a manifold and a metric
2. Defining tensors with various slot symmetries
3. What `SignedPerm` and `SlotSymmetry` are and how to work with them directly
4. How symmetry is stored on a tensor and what you can query
5. `canonical_rep` — the key algorithm that will power index sorting

---
## 1. Load package

In [1]:
using AbstractTensors

[ Info: Precompiling AbstractTensors [55c8b40c-cbfa-4141-803e-4831c9841971] (cache misses: include_dependency fsize change (6), mismatched flags (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


---
## 2. Define a manifold and a metric

`@def_manifold` creates the tangent and cotangent bundles and registers the indices.  
`@def_metric` defines a rank-2 covariant symmetric tensor and registers it in `_METRIC_REGISTRY`.

In [2]:
@def_manifold M 4 [a1, a2, a3, a4]

Defined manifold M of dimension 4 with tangent bundle tangentM and cotangent bundle cotangentM


In [3]:
?M

search: M



No documentation found for private binding `Main.M`.

`M` is of type `Manifold`.

# Summary

```
struct Manifold
```

# Fields

```
name             :: Symbol
dim              :: Int64
tangent_bundle   :: Symbol
cotangent_bundle :: Symbol
vbundles         :: Vector{Symbol}
```


In [4]:
@undef_manifold M

Undefined tangent bundle:   tangentM
Undefined cotangent bundle: cotangentM


┌ Warning: Manifold M has been undefined. The variable M still holds a stale reference,accessing it will error. Restart the kernel to fully clear the binding.
└ @ Main ~/JuliaTensor/JuliaTensorPackages/AbstractTensors/src/manifolds.jl:412


In [5]:
M.vbundles

┌ Warning: Manifold :M has been undefined. Variable still holds a stale reference.
└ @ AbstractTensors ~/JuliaTensor/JuliaTensorPackages/AbstractTensors/src/manifolds.jl:451


In [ ]:
@def_manifold M 4 [a1, a2, a3, a4]

In [6]:
@def_metric g[-a1, -a2] M

LoadError: @def_metric: manifold M is not registered. Call @def_manifold M first.

In [7]:
?N

search:

Couldn't find N
Perhaps you meant ∈, }, >, ≥, ⊽, ≈, ⊻, [, ÷, ∉, *, ∪, ⊋, ∌, ⊇, ≤, ?, (, +,  or π


No documentation found.

Binding `N` does not exist.


In [8]:
# Inspect what was created
tensor_info(g)

LoadError: UndefVarError: `g` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [9]:
_METRICS

Dict{Symbol, Symbol}()

In [10]:
metrics_of_manifold(:M)

Symbol[]

In [11]:
?M

search: M



No documentation found for private binding `Main.M`.

`M` is of type `Manifold`.

# Summary

```
struct Manifold
```

# Fields

```
name             :: Symbol
dim              :: Int64
tangent_bundle   :: Symbol
cotangent_bundle :: Symbol
vbundles         :: Vector{Symbol}
```


---
## 3. Define tensors with different symmetries

The `symmetries=` keyword accepts a `Vector{SlotSymmetry}`.  
Convenience constructors: `symmetric(n)`, `antisymmetric(n)`, `symmetric_on(n, pos)`, `antisymmetric_on(n, pos)`, `riemann_symmetry()`.

In [12]:
# No symmetry — generic rank-2 tensor
@def_tensor T[-a1, -a2] M metric=g
tensor_info(T)

LoadError: @def_tensor: manifold M is not registered. Call @def_manifold M first.

In [13]:
# Fully symmetric rank-2 (e.g. stress tensor)
@def_tensor S[-a1, -a2] M symmetries=[symmetric(2)] metric=g
tensor_info(S)

LoadError: @def_tensor: manifold M is not registered. Call @def_manifold M first.

In [14]:
# Fully antisymmetric rank-2 (e.g. electromagnetic field)
@def_tensor F[-a1, -a2] M symmetries=[antisymmetric(2)] metric=g
tensor_info(F)

LoadError: @def_tensor: manifold M is not registered. Call @def_manifold M first.

In [15]:
# Riemann tensor — standard algebraic symmetries
# R_{abcd} = -R_{bacd} = -R_{abdc} = R_{cdab}
@def_tensor R[-a1, -a2, -a3, a4] M symmetries=[riemann_symmetry()] metric=g
tensor_info(R)

LoadError: @def_tensor: manifold M is not registered. Call @def_manifold M first.

In [16]:
# Mixed symmetry: antisymmetric in slots (1,2), symmetric in slots (3,4)
@def_tensor B[-a1, -a2, -a3, -a4] M symmetries=[antisymmetric_on(4,[1,2]), symmetric_on(4,[3,4])] metric=g
tensor_info(B)

LoadError: @def_tensor: manifold M is not registered. Call @def_manifold M first.

---
## 4. Tensor expressions — the `T[...]` syntax

`T[...]` creates an inert `TensorExpression`. No computation happens yet.

In [17]:
# Covariant expression
g[-a1, -a2]

LoadError: UndefVarError: `g` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [18]:
# Antisymmetric tensor
F[-a1, -a2]

LoadError: UndefVarError: `F` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [19]:
# Riemann tensor
R[-a1, -a2, -a3, a4]

LoadError: UndefVarError: `R` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [20]:
?@def_tensor

```julia
@def_tensor name[slot1, slot2, ...] manifold
@def_tensor name[slot1, slot2, ...] manifold  symmetries=[S1,...]  traceless=bool  print_as=:sym  metric=g
```

Define a new abstract tensor on `manifold` and bind it to `name` in the caller's scope.

## Slot syntax

  * `-idx` : covariant (lower) slot — index lives in `cotangentM`
  * `idx` : contravariant (upper) slot — index lives in `tangentM`

The index symbols (`idx`) must already be registered to `manifold` via `@def_manifold` or `@add_indices`. They are used only for validation; the tensor stores vbundle names per slot, not the defining symbols.

## Keyword arguments (all optional)

  * `symmetries`  : a [`SlotSymmetry`](@ref) or `Vector{SlotSymmetry}` describing                 the slot permutation symmetry group(s). Defaults to                 `[no_symmetry(rank)]`. Use `symmetric(2)`, `antisymmetric(2)`,                 `symmetric_on(4, [1,2])`, `riemann_symmetry()`, etc.
  * `traceless`   : `true` if any self-contraction of this tensor is zero                 (e.g. Weyl tensor). Defaults to `false`.
  * `print_as`    : display name. Defaults to `name`. Example: `print_as=:R`.
  * `metric`      : name of the metric tensor to associate with this tensor.                 Omitting this keyword triggers automatic resolution:                 - one metric on manifold → assigned silently                 - multiple metrics → `@warn`, first defined is assigned                 - no metric → `@warn`, `nothing` assigned (no raising/lowering)

Binds `name` to a [`Tensor`](@ref) instance in the caller's scope and registers it in [`_TENSORS`](@ref).

# Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4]
@def_metric g[-a1, -a2] M

@def_tensor T[-a1, -a2] M                                   # metric auto-resolved to :g
@def_tensor F[-a1, -a2] M symmetries=[antisymmetric(2)]
@def_tensor R[-a1,-a2,-a3,-a4] M symmetries=[riemann_symmetry()]
@def_tensor W[-a1,-a2,-a3,-a4] M symmetries=[riemann_symmetry()] traceless=true print_as=:Weyl
@def_tensor mixed_T[a1, -a2] M   # mixed (1,1) tensor
```


In [21]:
# Mixed variance (one up, one down)
@def_tensor Gmix[a1, -a2] M metric=g print_as=G
Gmix[a1, -a2]

LoadError: @def_tensor: manifold M is not registered. Call @def_manifold M first.

In [22]:
Gmix.metric
Gmix[-a1, a2]

LoadError: UndefVarError: `Gmix` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

---
## 5. `SignedPerm` — permutation + sign

A `SignedPerm` represents one slot reordering. The sign `±1` is the scalar factor the tensor picks up.

In [23]:
# Identity on 4 slots
e4 = identity_perm(4)

SignedPerm([1, 2, 3, 4], +1)

In [24]:
# Swap positions 1 and 2, no sign change (symmetric transposition)
swap12_sym = SignedPerm([2, 1, 3, 4], Int8(1))

SignedPerm([2, 1, 3, 4], +1)

In [25]:
# Swap positions 1 and 2, flip sign (antisymmetric transposition)
swap12_anti = SignedPerm([2, 1, 3, 4], Int8(-1))

SignedPerm([2, 1, 3, 4], -1)

In [26]:
# Validate
is_valid_perm(swap12_sym), is_valid_perm(swap12_anti)

(true, true)

In [27]:
# Apply to a vector — reorders according to images
v = [:a, :b, :c, :d]
apply(swap12_anti, v)   # [:b, :a, :c, :d]  (sign not applied to the vector)

4-element Vector{Symbol}:
 :b
 :a
 :c
 :d

In [28]:
# Compose: f ∘ g means apply g first, then f
swap23 = SignedPerm([1, 3, 2, 4], Int8(-1))
fg = compose(swap12_anti, swap23)
println("swap12 ∘ swap23 = ", fg)
println("apply to [:a,:b,:c,:d] → ", apply(fg, v))

swap12 ∘ swap23 = SignedPerm([3, 1, 2, 4], +1)
apply to [:a,:b,:c,:d] → [:c, :a, :b, :d]


In [29]:
# Inverse
g_inv = inv(swap12_anti)
println("inv(swap12_anti) = ", g_inv)
id_check = compose(swap12_anti, g_inv)
println("swap12 ∘ inv(swap12) = ", id_check, "  (should be identity)")

inv(swap12_anti) = SignedPerm([2, 1, 3, 4], -1)
swap12 ∘ inv(swap12) = SignedPerm([1, 2, 3, 4], +1)  (should be identity)


---
## 6. `SlotSymmetry` — the full symmetry group of a tensor

A `SlotSymmetry` stores:
- the generators you provided
- **all group elements** (BFS closure computed at construction time)

This makes membership tests O(1) and `canonical_rep` a simple loop.

In [30]:
# Trivial symmetry
ns = no_symmetry(3)
println(ns)
println("group order = ", length(ns.group_elements))

NoSymmetry(n=3)
group order = 1


In [31]:
# Fully symmetric on 3 slots — group S3, order 3! = 6
sym3 = symmetric(3)
println(sym3)
println("group order = ", length(sym3.group_elements))
println("generators:")
for g in sym3.generators; println("  ", g); end

SlotSymmetry(n=3, order=6, ngens=2)
group order = 6
generators:
  SignedPerm([2, 1, 3], +1)
  SignedPerm([1, 3, 2], +1)


In [32]:
sym3.group_elements

6-element Vector{SignedPerm}:
 SignedPerm([3, 1, 2], +1)
 SignedPerm([2, 1, 3], +1)
 SignedPerm([3, 2, 1], +1)
 SignedPerm([1, 3, 2], +1)
 SignedPerm([2, 3, 1], +1)
 SignedPerm([1, 2, 3], +1)

In [33]:
# All elements of S3
println("All elements of symmetric(3):")
for el in sym3.group_elements
    println("  ", el)
end

All elements of symmetric(3):
  SignedPerm([3, 1, 2], +1)
  SignedPerm([2, 1, 3], +1)
  SignedPerm([3, 2, 1], +1)
  SignedPerm([1, 3, 2], +1)
  SignedPerm([2, 3, 1], +1)
  SignedPerm([1, 2, 3], +1)


In [34]:
# Fully antisymmetric on 3 slots — same S3 but each transposition carries sign -1
anti3 = antisymmetric(3)
println(anti3)
println("All elements:")
for el in anti3.group_elements
    println("  ", el)
end

SlotSymmetry(n=3, order=6, ngens=2)
All elements:
  SignedPerm([3, 1, 2], +1)
  SignedPerm([2, 1, 3], -1)
  SignedPerm([3, 2, 1], -1)
  SignedPerm([1, 3, 2], -1)
  SignedPerm([2, 3, 1], +1)
  SignedPerm([1, 2, 3], +1)


In [35]:
# Riemann symmetry: group order 8
riem = riemann_symmetry()
println(riem)
println("All ", length(riem.group_elements), " elements:")
for el in riem.group_elements
    println("  ", el)
end

SlotSymmetry(n=4, order=8, ngens=3)
All 8 elements:
  SignedPerm([2, 1, 4, 3], +1)
  SignedPerm([4, 3, 2, 1], +1)
  SignedPerm([2, 1, 3, 4], -1)
  SignedPerm([3, 4, 2, 1], -1)
  SignedPerm([1, 2, 3, 4], +1)
  SignedPerm([3, 4, 1, 2], +1)
  SignedPerm([1, 2, 4, 3], -1)
  SignedPerm([4, 3, 1, 2], -1)


In [36]:
# Membership test
pair_swap = SignedPerm([3, 4, 1, 2], Int8(1))   # R_{abcd} = R_{cdab}
println("pair swap in Riemann group? ", is_in_symmetry(pair_swap, riem))

wrong = SignedPerm([2, 1, 3, 4], Int8(1))        # +swap12 not in Riemann
println("wrong swap in Riemann group? ", is_in_symmetry(wrong, riem))

pair swap in Riemann group? true
wrong swap in Riemann group? false


In [37]:
# Partial symmetry: symmetric only on slots [1,2] of a rank-4 tensor
sym_on_12 = symmetric_on(4, [1, 2])
println(sym_on_12)
println("group order = ", length(sym_on_12.group_elements), "  (should be 2)")

SlotSymmetry(n=4, order=2, ngens=1)
group order = 2  (should be 2)


---
## 7. How symmetry is stored on a `Tensor`

The `Tensor` struct carries a `symmetries::Vector{SlotSymmetry}` field.  
Read it via `T.symmetries` (dot access on the tensor variable).

In [38]:
# Access symmetries stored on F
println("F symmetries: ", F.symmetries)
println("group order:  ", length(F.symmetries[1].group_elements))

LoadError: UndefVarError: `F` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [39]:
println("R symmetries: ", R.symmetries)
println("group order:  ", length(R.symmetries[1].group_elements))

LoadError: UndefVarError: `R` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [40]:
# is_trivial_symmetry
println("T has trivial symmetry? ", is_trivial_symmetry(T.symmetries[1]))
println("F has trivial symmetry? ", is_trivial_symmetry(F.symmetries[1]))

LoadError: UndefVarError: `T` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

---
## 8. `canonical_rep` — canonical index ordering under symmetry

`canonical_rep(indices, sym)` returns the lexicographically smallest index list reachable by the symmetry group, plus the sign of the group element that produced it.

This is the core algorithm for index canonicalization:  
`T[b, a] = sign × T[canonical_ordering]`

In [41]:
# Antisymmetric rank-2: F[a2, a1] = -F[a1, a2]
idx_a1 = down(:a1)   # TensorIndex(:a1, :CoTangentM)
idx_a2 = down(:a2)

canon, sign = canonical_rep([idx_a2, idx_a1], antisymmetric(2))
println("canonical indices: ", [i.symbol for i in canon])
println("sign: ", sign)
println("→  F[a2,a1] = ", sign, " × F[a1,a2]")

LoadError: Index :a1 is not registered. Was @def_manifold called?

In [42]:
# Symmetric rank-2: S[a2, a1] = +S[a1, a2]
canon, sign = canonical_rep([idx_a2, idx_a1], symmetric(2))
println("canonical indices: ", [i.symbol for i in canon])
println("sign: ", sign)
println("→  S[a2,a1] = ", sign, " × S[a1,a2]")

LoadError: UndefVarError: `idx_a2` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [43]:
# Riemann: R[a2,a1,a3,a4] = -R[a1,a2,a3,a4]
idx_a3 = down(:a3)
idx_a4 = down(:a4)

canon, sign = canonical_rep([idx_a2, idx_a1, idx_a3, idx_a4], riemann_symmetry())
println("canonical indices: ", [i.symbol for i in canon])
println("sign: ", sign)
println("→  R[a2,a1,a3,a4] = ", sign, " × R[canonical]")

LoadError: Index :a3 is not registered. Was @def_manifold called?

In [44]:
# Riemann pair-exchange: R[a3,a4,a1,a2] = +R[a1,a2,a3,a4]
canon, sign = canonical_rep([idx_a3, idx_a4, idx_a1, idx_a2], riemann_symmetry())
println("canonical indices: ", [i.symbol for i in canon])
println("sign: ", sign)
println("→  R[a3,a4,a1,a2] = ", sign, " × R[canonical]")

LoadError: UndefVarError: `idx_a3` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

---
## 9. Registry inspection

All defined tensors and metrics are registered globally.

In [45]:
println("Registered tensors: ", collect(keys(_TENSOR_REGISTRY)))

LoadError: UndefVarError: `_TENSOR_REGISTRY` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [46]:
println("Registered metrics: ", collect(keys(_METRIC_REGISTRY)))
println()
list_metrics()

LoadError: UndefVarError: `_METRIC_REGISTRY` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [47]:
println("Metrics on manifold M: ", metrics_on_manifold(:M))

LoadError: UndefVarError: `metrics_on_manifold` not defined in `Main`
Suggestion: check for spelling errors or missing imports.